# Valuation of investment

In [144]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from dotenv import load_dotenv
import os
import sys

In [28]:
load_dotenv()  # carga .env
sys.path.insert(0, os.path.join(os.environ["CUANTIS_ROOT"], "src"))

In [29]:
from cuantis_utils.TestDistribuitions import best_fit_distribution

## HW: 3.1

Buy a start up stock that doesn't pay dividens estimate the expected results using Corporate Finance Theory the $\text{X}$ factor stochastic.

Assume a normal distribution

- Analytical method
- Simulate in excel/python to test accuracy

### Analytical Derivation

Assume the stochastic factor $X$ is normally distributed:

$$
X \sim \mathcal{N}(\mu_X, \sigma_X^2).
$$

Using the corporate finance identity (treating $A$, $L$, and $N$ as constants):

$$
X = \frac{P_x N + L}{A} \quad \Rightarrow \quad P_x = \frac{A X - L}{N}.
$$

Then the expected value and variance of $P_x$ are:

$$
\operatorname{E}[P_x] = \operatorname{E}\left[\frac{A X - L}{N}\right]
= \frac{A}{N}\operatorname{E}[X] - \frac{L}{N}
= \frac{A}{N}\mu_X - \frac{L}{N}.
$$

$$
\operatorname{Var}(P_x) = \operatorname{Var}\left(\frac{A}{N}X - \frac{L}{N}\right)
= \left(\frac{A}{N}\right)^2 \operatorname{Var}(X)
= \left(\frac{A}{N}\right)^2 \sigma_X^2.
$$

Since $P_x$ is an affine transformation of a normal random variable, it is also normal:

$$
P_x \sim \mathcal{N}\left(\frac{A}{N}\mu_X - \frac{L}{N},\, \left(\frac{A}{N}\right)^2 \sigma_X^2\right).
$$


### Simulate in python

In [ ]:
A = 225_248e6       # ejemplo (en USD) ajusta unidades
L = 163_188e6
N = 497.8e6

mu_X = 6.5
sigma_X = 0.8       # ejemplo: tú lo defines/justificas
M = 200_000

X = np.random.normal(mu_X, sigma_X, size=M)
P = (A * X - L) / N

E_P = (A * mu_X - L) / N
Var_P = (A/N)**2 * sigma_X**2

mc_mean = P.mean()
mc_var = P.var(ddof=0)

mean_abs_err = abs(mc_mean - E_P)
var_abs_err = abs(mc_var - Var_P)

mean_rel_err = mean_abs_err / abs(E_P) if E_P != 0 else np.nan
var_rel_err = var_abs_err / abs(Var_P) if Var_P != 0 else np.nan

print(f'Monte Carlo check (M={M:,})')
print(f"E[P_x]   MC={mc_mean:,.6f} | Analytical={E_P:,.6f} | abs err={mean_abs_err:,.6f} | rel err={mean_rel_err:.4%}")
print(f"Var[P_x] MC={mc_var:,.6f} | Analytical={Var_P:,.6f} | abs err={var_abs_err:,.6f} | rel err={var_rel_err:.4%}")

## HW: 3.2

Using the properties of random variables and knowing the PDF of the sales product of each year answer analytically:

#### What are the expected product sales for year 1 and year 2? (Use ML code)

In [154]:
df = pd.read_csv("../data/01_Data_OilCompany.csv")

In [155]:
df.head(3)

,Year 1,Year 2,Year 3,Year 4,Year 5
0,"203,726.00","279,969.00","465,303.00","200,445.00","96,319.00"
1,"263,845.00","272,439.00","2,192.00","199,901.00",0.00
2,"27,726.00","334,861.00","82,265.00","199,954.00","64,958.00"


In [156]:
year_1 = df.iloc[:, [0]]
year_2 = df.iloc[:, [1]]

In [157]:
best_fit_name, best_fit_params = best_fit_distribution(year_1)
best_fit_name_2, best_fit_params_2 = best_fit_distribution(year_2)

In [148]:
print(f"Year 1: {best_fit_params[0]}")
print(f"Year 2: {best_fit_params[0]}")

Year 1: 196593.614
Year 2: 196593.614


#### What the Expected Value and Var of IRR and NPV


In [158]:
from cuantis_utils.InvestmentValuation import valuate_investment

In [172]:
np.std(df.sum(axis=1))

np.float64(224874.85428026333)

In [173]:
initial_investment = 850_00 
discount_rate = 0.12

In [174]:
results = valuate_investment(df, -initial_investment, discount_rate)

In [175]:
results.head()

,NPV,IRR
0,"1,028,520.28",2.73
1,"651,127.14",2.95
2,"575,895.48",1.37
3,"843,012.21",3.25
4,"1,116,444.62",3.45


In [ ]:
dist_IRR_name, params_IRR_dist = best_fit_distribution(results["IRR"])
dist_NPV_name, params_NPV_dist = best_fit_distribution(results["NPV"])

In [181]:
print(f"expected value: {params_IRR_dist[0]:,.4f}")
print(f"Var: {params_IRR_dist[1]**2:,.4f}")

expected value: 0.0452
Var: 0.0091


In [180]:
print(f"expected value: {params_NPV_dist[0]:,.2f}")
print(f"Var: {params_NPV_dist[1]**2:,.2f}")

expected value: 802,671.81
Var: 35,085,405,878.87


In [ ]:
params_IRR_dist

**NPV**:
- expected value: 802,671.81
- Var: 35,102,957,357.55

**IRR**:
- expected value: 0.0452
- Var: 0.0091

#### What value of expected value and Var for NPV

Let the cash flows (product sales) for each year be random variables $CF_1,\dots,CF_5$, and let the risk-free discount rate be $r_f = 0.12$.

In this notebook, the NPV is defined as a discounted sum with $\text{Year 1}$ at $t=0$:

$$
NPV = \sum_{t=0}^{4} \frac{CF_{t+1}}{(1+r_f)^t}.
$$

Because $NPV$ is a linear combination of random variables, its first two moments follow directly:

$$
\operatorname{E}[NPV] = \sum_{t=0}^{4} \frac{\operatorname{E}[CF_{t+1}]}{(1+r_f)^t}.
$$

$$
\operatorname{Var}(NPV)
= \sum_{t=0}^{4} \frac{\operatorname{Var}(CF_{t+1})}{(1+r_f)^{2t}}
+ 2\sum_{0\le i<j\le 4} \frac{\operatorname{Cov}(CF_{i+1},CF_{j+1})}{(1+r_f)^{i+j}}.
$$

If you assume independence across years, the covariance terms are zero.

Using $n=5$ and $r_f=0.12$, the estimated values (from the results in this notebook) are:

$$
\operatorname{E}[NPV] = 802{,}671.81.
$$

$$
\operatorname{Var}(NPV) = 35{,}085{,}405{,}878.87.
$$


#### Estimate the PDF of NPV and IRR and answer analytically

In [182]:
mu_IRR = params_IRR_dist[0]
sigma_IRR = params_IRR_dist[1]

p_IRR = stats.norm(mu_IRR, sigma_IRR)

mu_NPV = params_NPV_dist[0]
sigma_NPV = params_NPV_dist[1]

p_NPV = stats.norm(mu_NPV, sigma_NPV)

What the probability that the IRR is over the Risk free rate?

In [183]:
p_IRR.sf(discount_rate)

np.float64(0.21645228476860395)

What the probability that the IRR is over the 35%?

In [184]:
p_IRR.sf(0.35)

np.float64(0.0007006494255711977)

What the probability that the project value is over $2M?


In [186]:
p_NPV.sf(2_000_000)

np.float64(8.176104389178706e-11)

What’s the probability that the IRR is between 10% and 20%?


In [187]:
p_IRR.sf(0.1) - p_IRR.sf(0.2)

np.float64(0.23041978746419128)

Do you get same responses as in Homework 1.1? NO